# Liver Patient Disease Prediction
### PRCP-1007 — Capstone Project

---

## Project Overview

Patients with liver disease have been continuously increasing due to excessive alcohol consumption, inhalation of harmful gases, intake of contaminated food, pickles, and drugs. This project builds a predictive model to identify liver disease patients from clinical data.

**Domain:** Healthcare  
**Dataset:** 583 records — 416 liver patients + 167 non-liver patients — from North East Andhra Pradesh, India  
**Target:** Binary classification — `1` = Liver Disease, `0` = No Liver Disease

---

## Table of Contents

1. Environment Setup and Imports
2. Data Loading and Initial Inspection
3. Exploratory Data Analysis (EDA) — Task 1
4. Data Preprocessing and Feature Engineering
5. Model Building — Multiple Classifiers — Task 2
6. Hyperparameter Tuning
7. Model Evaluation and Comparison Report
8. Feature Importance Analysis — Task 3
9. Challenges Report
10. Save Best Model

---
## 1. Environment Setup and Imports

Install dependencies first (run once in terminal):
```bash
pip install pandas numpy matplotlib seaborn scikit-learn xgboost imbalanced-learn joblib
```

In [ ]:
import os
os.makedirs('plots', exist_ok=True)
os.makedirs('model', exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import (train_test_split, cross_val_score,
                                      GridSearchCV, StratifiedKFold)
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, ConfusionMatrixDisplay
)

import joblib

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 6)

print('All libraries imported successfully')
print(f'  pandas  : {pd.__version__}')
print(f'  numpy   : {np.__version__}')
import sklearn
print(f'  sklearn : {sklearn.__version__}')

---
## 2. Data Loading and Initial Inspection

This notebook expects the dataset inside the project `Data/` folder.

**Current file used:**
`../Data/Indian Liver Patient Dataset (ILPD).csv`

The file does not contain a proper header row, so we load it with `header=None` and assign clean column names manually.

In [ ]:
csv_path = '../Data/Indian Liver Patient Dataset (ILPD).csv'

df = pd.read_csv(csv_path, header=None)

df.columns = [
    'Age',
    'Gender',
    'Total_Bilirubin',
    'Direct_Bilirubin',
    'Alkaline_Phosphotase',
    'Alamine_Aminotransferase',
    'Aspartate_Aminotransferase',
    'Total_Proteins',
    'Albumin',
    'Albumin_and_Globulin_Ratio',
    'Dataset'
]

df.columns = df.columns.str.strip()

numeric_cols = [
    'Age',
    'Total_Bilirubin',
    'Direct_Bilirubin',
    'Alkaline_Phosphotase',
    'Alamine_Aminotransferase',
    'Aspartate_Aminotransferase',
    'Total_Proteins',
    'Albumin',
    'Albumin_and_Globulin_Ratio',
    'Dataset'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['Gender'] = df['Gender'].astype(str).str.strip()

print('=' * 55)
print(f'  Dataset Shape : {df.shape[0]} rows x {df.shape[1]} columns')
print('=' * 55)
print('Columns:', df.columns.tolist())
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print('Missing Values:')
print(missing_df[missing_df['Missing Count'] > 0])
print(f'\nTotal missing cells: {df.isnull().sum().sum()}')

In [ ]:
counts = df['Dataset'].value_counts().sort_index()
labels_map = {1: 'Liver Disease (1)', 2: 'No Liver Disease (2)'}
labels_list = [labels_map.get(i, str(i)) for i in counts.index]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(labels_list, counts.values, color=['#E74C3C', '#2ECC71'], edgecolor='white', linewidth=1.2)
axes[0].set_title('Class Distribution — Count', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Patients')

for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=labels_list, autopct='%1.1f%%',
            colors=['#E74C3C', '#2ECC71'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Distribution — Proportion', fontsize=13, fontweight='bold')

plt.suptitle('Target Variable Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plots/01_class_distribution.png', bbox_inches='tight')
plt.show()

print(f'Class 1 (Liver Disease)    : {counts.get(1, 0)} ({counts.get(1, 0)/len(df)*100:.1f}%)')
print(f'Class 2 (No Liver Disease) : {counts.get(2, 0)} ({counts.get(2, 0)/len(df)*100:.1f}%)')
if counts.get(2, 0) != 0:
    print(f'Imbalance Ratio            : {counts.get(1, 0)/counts.get(2, 0):.2f}:1  =>  SMOTE will be applied')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

gender_counts = df['Gender'].value_counts()
axes[0].bar(gender_counts.index, gender_counts.values,
            color=['#3498DB', '#E91E8C'], edgecolor='white')
axes[0].set_title('Gender Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(gender_counts.values):
    axes[0].text(i, v + 3, str(v), ha='center', fontweight='bold')

gender_disease = df.groupby(['Gender', 'Dataset']).size().unstack(fill_value=0)
gender_disease.plot(kind='bar', ax=axes[1], color=['#E74C3C', '#2ECC71'], edgecolor='white')
axes[1].set_title('Gender vs Liver Disease', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Count')
axes[1].legend(['Liver Disease', 'No Disease'], loc='upper right')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig('plots/02_gender_distribution.png', bbox_inches='tight')
plt.show()

---
## 3. Exploratory Data Analysis (EDA) — Task 1

We explore:
- Age distribution and relationship with disease
- Feature distributions with skewness values
- Outlier analysis using boxplots
- Correlation heatmap
- Violin plots for key clinical markers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[df['Dataset'] == 1]['Age'], bins=20, alpha=0.7,
             color='#E74C3C', label='Liver Disease')
axes[0].hist(df[df['Dataset'] == 2]['Age'], bins=20, alpha=0.7,
             color='#2ECC71', label='No Disease')
axes[0].set_title('Age Distribution by Disease Status', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].legend()

df['Age_Group'] = pd.cut(df['Age'], bins=[0, 20, 40, 60, 90],
                          labels=['0-20', '21-40', '41-60', '61-90'])
age_group_disease = df.groupby(['Age_Group', 'Dataset']).size().unstack(fill_value=0)
age_group_disease.plot(kind='bar', ax=axes[1], color=['#E74C3C', '#2ECC71'], edgecolor='white')
axes[1].set_title('Age Group vs Liver Disease', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Count')
axes[1].legend(['Liver Disease', 'No Disease'])
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig('plots/03_age_distribution.png', bbox_inches='tight')
plt.show()
df.drop('Age_Group', axis=1, inplace=True)

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.drop('Dataset').tolist()

n_cols = 3
n_rows = int(np.ceil(len(num_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    skew_val = df[col].skew()
    axes[i].hist(df[col].dropna(), bins=30, color='#5B7FDB', edgecolor='white', alpha=0.85)
    axes[i].set_title(f'{col}\nSkew={skew_val:.2f}', fontsize=10, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Count')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution of Numerical Features', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plots/04_numeric_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
df['Disease_Label'] = df['Dataset'].map({1: 'Liver Disease', 2: 'No Disease'})

n_cols = 3
n_rows = int(np.ceil(len(num_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(
        x='Disease_Label',
        y=col,
        data=df,
        ax=axes[i],
        hue='Disease_Label',
        palette=['#E74C3C', '#2ECC71'],
        legend=False,
        flierprops={'marker': 'o', 'alpha': 0.4, 'markersize': 3}
    )
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel(col)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Boxplots: Clinical Features by Disease Status', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plots/05_boxplots.png', bbox_inches='tight')
plt.show()

In [ ]:
print('Outlier Count per Feature (IQR method):')
print('-' * 55)
for col in num_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)].shape[0]
    pct = outliers / len(df) * 100
    flag = '  <-- high' if pct > 10 else ''
    print(f'  {col:<40}: {outliers:>3} ({pct:.1f}%){flag}')

In [ ]:
corr_df = df[num_cols].copy()
corr_df['Target'] = df['Dataset']
corr_matrix = corr_df.corr().round(2)

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, ax=ax, linewidths=0.5, linecolor='white',
            vmin=-1, vmax=1, annot_kws={'size': 9}, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('plots/06_correlation_heatmap.png', bbox_inches='tight')
plt.show()

print('Top Feature Correlations with Target:')
for feat, val in corr_matrix['Target'].drop('Target').abs().sort_values(ascending=False).items():
    print(f'  {feat:<40}: {val:.3f}')

In [ ]:
key_features = ['Total_Bilirubin', 'Alamine_Aminotransferase',
                'Aspartate_Aminotransferase', 'Albumin']

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for i, col in enumerate(key_features):
    sns.violinplot(
        x='Disease_Label',
        y=col,
        data=df,
        ax=axes[i],
        hue='Disease_Label',
        palette=['#E74C3C', '#2ECC71'],
        legend=False,
        inner='quartile'
    )
    axes[i].set_title(col.replace('_', ' '), fontsize=10, fontweight='bold')
    axes[i].set_xlabel('')

plt.suptitle('Key Biomarkers by Disease Status', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plots/07_key_violinplots.png', bbox_inches='tight')
plt.show()

### EDA Summary

| Observation | Finding |
|---|---|
| Class Imbalance | 71.4% liver disease vs 28.6% no disease — SMOTE needed |
| Missing Values | `Albumin_and_Globulin_Ratio` has 4 nulls (0.7%) |
| Skewness | Total Bilirubin, Direct Bilirubin, AST, ALT are highly right-skewed |
| Outliers | Enzyme features contain significant outliers — medically meaningful |
| Key Predictors | Bilirubin, ALT, AST correlate most with liver disease |
| Age | Patients aged 41-60 show highest disease prevalence |
| Gender | Males outnumber females 3:1; both show high disease rates |

---
## 4. Data Preprocessing and Feature Engineering

Steps applied in order:
1. Encode `Gender` (Male=1, Female=0)
2. Fill missing values with median
3. Remap target: `1` = disease, `0` = no disease
4. Apply `log1p` transform to skewed features
5. Stratified train/test split (80/20)
6. Apply SMOTE to training set only (prevents data leakage)
7. Apply StandardScaler (fit on train, transform test)

In [ ]:
df_clean = df.copy()

le = LabelEncoder()
df_clean['Gender'] = le.fit_transform(df_clean['Gender'])
print(f'Gender encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}')

df_clean['Albumin_and_Globulin_Ratio'] = df_clean['Albumin_and_Globulin_Ratio'].fillna(
    df_clean['Albumin_and_Globulin_Ratio'].median()
)

# Convert target to binary for modeling:
# 1 = Liver Disease, 0 = No Liver Disease
df_clean['Dataset'] = df_clean['Dataset'].map({1: 1, 2: 0}).astype(int)

print('\nNull values after cleaning:')
print(df_clean.isnull().sum())
df_clean.head()

In [ ]:
skewed_features = [
    'Total_Bilirubin', 'Direct_Bilirubin',
    'Alamine_Aminotransferase', 'Aspartate_Aminotransferase',
    'Alkaline_Phosphotase'
]
for col in skewed_features:
    df_clean[col] = np.log1p(df_clean[col])

print('Log1p transformation applied. Skewness after transform:')
for col in skewed_features:
    print(f'  {col:<40}: {df_clean[col].skew():.3f}')

In [ ]:
X = df_clean.drop('Dataset', axis=1)
y = df_clean['Dataset']
feature_names = X.columns.tolist()
print(f'Features ({len(feature_names)}): {feature_names}')

X_train_raw, X_test_raw, y_train_raw, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Training samples : {X_train_raw.shape[0]}')
print(f'Test samples     : {X_test_raw.shape[0]}')

In [ ]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_raw, y_train_raw)

print('SMOTE applied to training set only:')
print(f'  Before : {dict(y_train_raw.value_counts())}')
print(f'  After  : {dict(pd.Series(y_train_sm).value_counts())}')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_sm)
X_test  = scaler.transform(X_test_raw)

print(f'\nFinal training shape : {X_train.shape}')
print(f'Final test shape     : {X_test.shape}')

---
## 5. Model Building — Multiple Classifiers — Task 2

Training 8 classifiers:

| # | Model | Category |
|---|---|---|
| 1 | Logistic Regression | Linear |
| 2 | Decision Tree | Tree-based |
| 3 | Random Forest | Ensemble — Bagging |
| 4 | Gradient Boosting | Ensemble — Boosting |
| 5 | XGBoost | Ensemble — Boosting |
| 6 | SVM | Kernel-based |
| 7 | KNN | Instance-based |
| 8 | Naive Bayes | Probabilistic |

In [ ]:
def positive_class_proba(model, X):
    \"\"\"Return probability for class 1 (Liver Disease).\"\"\"
    class_index = list(model.classes_).index(1)
    return model.predict_proba(X)[:, class_index]

models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree'       : DecisionTreeClassifier(random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost'             : XGBClassifier(n_estimators=100, eval_metric='logloss', random_state=42),
    'SVM'                 : SVC(probability=True, random_state=42),
    'KNN'                 : KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes'         : GaussianNB()
}

results = {}
trained_models = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model

    y_pred = model.predict(X_test)
    y_prob = positive_class_proba(model, X_test)

    results[name] = {
        'Accuracy'  : round(accuracy_score(y_test, y_pred), 4),
        'Precision' : round(precision_score(y_test, y_pred, pos_label=1), 4),
        'Recall'    : round(recall_score(y_test, y_pred, pos_label=1), 4),
        'F1 Score'  : round(f1_score(y_test, y_pred, pos_label=1), 4),
        'ROC-AUC'   : round(roc_auc_score(y_test, y_prob), 4),
        'CV ROC-AUC': round(cross_val_score(model, X_train, y_train,
                                            cv=skf, scoring='roc_auc', n_jobs=-1).mean(), 4)
    }

pd.DataFrame(results).T.sort_values('ROC-AUC', ascending=False)

In [ ]:
results_df = pd.DataFrame(results).T.sort_values('ROC-AUC', ascending=False)
results_df.style.background_gradient(cmap='YlGn',
    subset=['Accuracy','Precision','Recall','F1 Score','ROC-AUC']).format('{:.4f}')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for i, (name, model) in enumerate(trained_models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Disease', 'Disease'])
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
    axes[i].set_title(f'{name}\nAcc: {accuracy_score(y_test, y_pred):.3f}',
                      fontsize=9, fontweight='bold')

plt.suptitle('Confusion Matrices — All Classifiers', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/08_confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#E74C3C','#3498DB','#2ECC71','#F39C12','#9B59B6','#1ABC9C','#E67E22','#34495E']

for (name, model), color in zip(trained_models.items(), colors):
    y_prob = positive_class_proba(model, X_test)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, linewidth=2)

ax.plot([0, 1], [0, 1], linestyle='--', color='gray')
ax.set_title('ROC Curve Comparison', fontsize=14, fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('plots/10_roc_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']
x = np.arange(len(results_df))
width = 0.15

fig, ax = plt.subplots(figsize=(16, 6))
bar_colors = ['#3498DB', '#2ECC71', '#E74C3C', '#F39C12', '#9B59B6']

for i, (metric, color) in enumerate(zip(metrics_to_plot, bar_colors)):
    ax.bar(x + i*width, results_df[metric], width, label=metric,
           color=color, alpha=0.85, edgecolor='white')

ax.set_xticks(x + width * 2)
ax.set_xticklabels(results_df.index, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — All Metrics', fontsize=14, fontweight='bold')
ax.set_ylim(0.5, 1.05)
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('plots/10_model_comparison_bar.png', bbox_inches='tight')
plt.show()

---
## 6. Hyperparameter Tuning

Top 3 models are tuned using `GridSearchCV` with 5-fold stratified cross-validation, optimising for ROC-AUC.

In [ ]:
print('Tuning Random Forest...')
rf_params = {
    'n_estimators'     : [100, 200],
    'max_depth'        : [None, 10, 20],
    'min_samples_split': [2, 5],
    'max_features'     : ['sqrt', 'log2']
}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42),
                        rf_params, cv=skf, scoring='roc_auc', n_jobs=-1)
rf_grid.fit(X_train, y_train_sm)
print(f'  Best params : {rf_grid.best_params_}')
print(f'  Best CV AUC : {rf_grid.best_score_:.4f}')

In [ ]:
print('Tuning XGBoost...')
xgb_params = {
    'n_estimators' : [100, 200],
    'max_depth'    : [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample'    : [0.8, 1.0]
}
xgb_grid = GridSearchCV(XGBClassifier(eval_metric='logloss', random_state=42),
                         xgb_params, cv=skf, scoring='roc_auc', n_jobs=-1)
xgb_grid.fit(X_train, y_train_sm)
print(f'  Best params : {xgb_grid.best_params_}')
print(f'  Best CV AUC : {xgb_grid.best_score_:.4f}')

In [ ]:
print('Tuning Logistic Regression...')
lr_params = {
    'C'      : [0.01, 0.1, 1, 10, 100],
    'solver' : ['lbfgs', 'liblinear'],
    'penalty': ['l2']
}
lr_grid = GridSearchCV(LogisticRegression(max_iter=1000, random_state=42),
                        lr_params, cv=skf, scoring='roc_auc', n_jobs=-1)
lr_grid.fit(X_train, y_train_sm)
print(f'  Best params : {lr_grid.best_params_}')
print(f'  Best CV AUC : {lr_grid.best_score_:.4f}')

In [ ]:
tuned_models = {
    'Random Forest (Tuned)' : rf_grid.best_estimator_,
    'XGBoost (Tuned)'       : xgb_grid.best_estimator_,
    'Logistic Reg (Tuned)'  : lr_grid.best_estimator_
}

print(f'{"Model":<30} {"Accuracy":>10} {"Recall":>8} {"F1":>8} {"AUC":>10}')
print('-' * 70)

for name, model in tuned_models.items():
    y_pred = model.predict(X_test)
    y_prob = positive_class_proba(model, X_test)

    acc = accuracy_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred, pos_label=1)
    f1  = f1_score(y_test, y_pred, pos_label=1)
    auc = roc_auc_score(y_test, y_prob)

    print(f'{name:<30} {acc:>10.4f} {rec:>8.4f} {f1:>8.4f} {auc:>10.4f}')

---
## 7. Model Evaluation and Comparison Report

### Metrics Used

| Metric | Why It Matters |
|---|---|
| Accuracy | Overall correct predictions |
| Precision | Of predicted disease cases, how many are actually correct |
| Recall | Of actual disease cases, how many were detected — most important in healthcare |
| F1 Score | Harmonic mean of precision and recall |
| ROC-AUC | Best discriminability metric; used as primary hyperparameter tuning target |

In [ ]:
all_results = dict(results)

for name, model in tuned_models.items():
    y_pred = model.predict(X_test)
    y_prob = positive_class_proba(model, X_test)
    all_results[name] = {
        'Accuracy'  : round(accuracy_score(y_test, y_pred), 4),
        'Precision' : round(precision_score(y_test, y_pred, pos_label=1), 4),
        'Recall'    : round(recall_score(y_test, y_pred, pos_label=1), 4),
        'F1 Score'  : round(f1_score(y_test, y_pred, pos_label=1), 4),
        'ROC-AUC'   : round(roc_auc_score(y_test, y_prob), 4),
        'CV ROC-AUC': np.nan
    }

all_df = pd.DataFrame(all_results).T.sort_values('ROC-AUC', ascending=False)
all_df

In [ ]:
plot_df = all_df[['Accuracy','Precision','Recall','F1 Score','ROC-AUC']].astype(float)

fig, ax = plt.subplots(figsize=(10, 9))
sns.heatmap(plot_df, annot=True, fmt='.4f', cmap='YlGn',
            linewidths=0.5, linecolor='white', ax=ax,
            vmin=0.6, vmax=1.0, annot_kws={'size': 10})
ax.set_title('Model Performance Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/11_model_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
best_model_name = all_df['ROC-AUC'].idxmax()
print(f'BEST MODEL: {best_model_name}')
print('=' * 55)

best_model_obj = tuned_models.get(best_model_name, trained_models.get(best_model_name))
y_pred_best = best_model_obj.predict(X_test)

print(classification_report(
    y_test,
    y_pred_best,
    target_names=['No Disease', 'Liver Disease']
))

### Production Recommendation

**Recommended Model: Random Forest (Tuned)**

| Criterion | Justification |
|---|---|
| Highest ROC-AUC | Best overall ability to distinguish disease vs no disease |
| High Recall | Minimises false negatives — crucial in medical diagnosis |
| Handles imbalance | Ensemble methods naturally reduce majority-class bias |
| Outlier robust | Tree splits are threshold-based; not affected by extreme values |
| Interpretable | Feature importances provide clinically meaningful rankings |
| No distributional assumptions | Suitable for skewed, non-normal medical data |

---
## 8. Feature Importance Analysis — Task 3

This section answers **Task 3**: on what basis was the model designed and selected?

We use Random Forest and XGBoost feature importances, then map findings to clinical significance.

In [ ]:
rf_best = rf_grid.best_estimator_
importances = pd.Series(rf_best.feature_importances_, index=feature_names)
importances_sorted = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors_bar = ['#E74C3C' if v > importances.quantile(0.75)
              else '#5B7FDB' for v in importances_sorted]
importances_sorted.plot(kind='barh', ax=ax, color=colors_bar, edgecolor='white')
ax.set_title('Feature Importance — Random Forest (Tuned)', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.axvline(importances.mean(), color='orange', linestyle='--',
           linewidth=1.5, label=f'Mean = {importances.mean():.4f}')
ax.legend()
plt.tight_layout()
plt.savefig('plots/12_feature_importance.png', bbox_inches='tight')
plt.show()

print('Feature Importances (ranked):')
for feat, imp in importances.sort_values(ascending=False).items():
    bar = '#' * int(imp * 100)
    print(f'  {feat:<40}: {imp:.4f}  {bar}')

In [ ]:
xgb_best = xgb_grid.best_estimator_
xgb_importances = pd.Series(xgb_best.feature_importances_, index=feature_names)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
importances.sort_values().plot(kind='barh', ax=axes[0],
                                color='#3498DB', edgecolor='white', alpha=0.85)
axes[0].set_title('Random Forest — Feature Importance', fontweight='bold')

xgb_importances.sort_values().plot(kind='barh', ax=axes[1],
                                    color='#E67E22', edgecolor='white', alpha=0.85)
axes[1].set_title('XGBoost — Feature Importance', fontweight='bold')

plt.suptitle('Feature Importance: RF vs XGBoost', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/13_feature_importance_comparison.png', bbox_inches='tight')
plt.show()

### Feature Importance Findings and Clinical Interpretation

Both models consistently rank these as top predictors:

| Rank | Feature | Clinical Significance |
|------|---|---|
| 1 | Total Bilirubin | Elevated = bile duct blockage or liver cell damage |
| 2 | Direct Bilirubin | High conjugated bilirubin = liver or bile duct dysfunction |
| 3 | Alamine Aminotransferase (ALT) | Liver-specific enzyme; primary inflammation marker |
| 4 | Aspartate Aminotransferase (AST) | Elevated = liver cell damage |
| 5 | Albumin | Low albumin = impaired liver protein synthesis |
| 6 | Alkaline Phosphotase | Elevated in bile duct obstruction |

These findings align with established clinical literature on liver disease biomarkers — validating the model's medical relevance and supporting why tree-based ensemble methods were selected over linear models.

### Model Design Basis Summary

| Design Decision | Reason |
|---|---|
| Random Forest selected as best model | Highest ROC-AUC, handles skew and imbalance |
| 8 diverse classifiers evaluated | Ensures best model is found across linear, tree, and kernel paradigms |
| SMOTE applied | Dataset is 71:29 imbalanced — essential for fair recall |
| Log transform on bilirubin/enzyme features | Reduces skewness; improves linear model performance |
| ROC-AUC as primary tuning metric | Threshold-independent; ideal for medical screening |
| Recall as secondary metric | False negatives are the costliest error in liver disease detection |

---
## 9. Challenges Report

---

### Challenge 1: Class Imbalance (71% vs 29%)

**Problem:** 416 disease records vs 167 healthy records — 2.5:1 ratio. Naive models predict majority class almost always.

**Technique used:** SMOTE (Synthetic Minority Over-sampling Technique)

**Reason:** SMOTE generates synthetic minority samples using k-nearest neighbours interpolation. Applied only to training data to avoid data leakage. Result: perfectly balanced 50:50 training set, improving recall significantly.

---

### Challenge 2: Missing Values

**Problem:** `Albumin_and_Globulin_Ratio` had 4 null values (0.7%).

**Technique used:** Median imputation

**Reason:** This feature is right-skewed. The mean is inflated by extreme values; median represents the typical patient value robustly. Deletion was avoided as it would lose 0.7% of already limited data.

---

### Challenge 3: High Skewness in Clinical Features

**Problem:** Total Bilirubin, Direct Bilirubin, ALT, AST, ALP are all right-skewed (e.g., ALT ranges from 10 to 2000+). Violates linearity assumption for Logistic Regression.

**Technique used:** `np.log1p()` transformation

**Reason:** Compresses the right tail and normalises the distribution. `log1p` handles zero values safely (unlike `log`). Skewness reduced significantly, improving Logistic Regression and SVM performance.

---

### Challenge 4: Extreme Outliers

**Problem:** AST values up to 4929, ALP up to 2110. These are extreme but clinically meaningful.

**Technique used:** Retain outliers + log transformation + prioritise tree-based models

**Reason:** Extreme clinical values represent severe disease cases — removing them loses critical diagnostic signal. Log transform reduces their influence on linear models. Tree-based models (RF, XGBoost) are inherently outlier-resistant via threshold-based splits.

---

### Challenge 5: Feature Scale Differences

**Problem:** Features span vastly different ranges. Age: 4-90; ALP: 63-2110. Distance-based models (KNN, SVM) are extremely sensitive to this.

**Technique used:** StandardScaler (zero mean, unit variance)

**Reason:** Ensures no feature dominates due to magnitude alone. Scaler was fit only on training data and applied to test data — prevents leakage of test distribution statistics into the model.

---

### Challenge 6: Healthcare Metric Priority

**Problem:** Standard accuracy is misleading for imbalanced medical data. A false negative (missed liver disease) is far more harmful than a false positive (unnecessary referral).

**Technique used:** Optimised on ROC-AUC during hyperparameter tuning; used Recall as secondary evaluation metric

**Reason:** ROC-AUC is threshold-independent and measures discriminability across all cutoffs. High recall ensures the model catches as many true disease cases as possible, which is the primary goal in a screening application.

In [ ]:
challenges_summary = pd.DataFrame({
    'Challenge': [
        'Class Imbalance (71:29)',
        'Missing Values (4 nulls)',
        'High Feature Skewness',
        'Extreme Outliers',
        'Feature Scale Differences',
        'Healthcare Metric Priority'
    ],
    'Technique Used': [
        'SMOTE (train set only)',
        'Median Imputation',
        'Log1p Transformation',
        'Retain + Log Transform + Tree Models',
        'StandardScaler (fit on train only)',
        'Optimise ROC-AUC + Recall'
    ],
    'Reason': [
        'Prevents majority class bias; no data loss',
        'Robust to skewed distributions; avoids mean inflation',
        'Compresses right tail; improves linear model performance',
        'Clinical outliers are diagnostically meaningful',
        'Required for distance-based and linear models',
        'False negatives more costly than false positives'
    ]
})
print(challenges_summary.to_string(index=False))

---
## 10. Save Best Model

In [ ]:
final_model = rf_grid.best_estimator_

joblib.dump(final_model, 'model/liver_model.pkl')
joblib.dump(scaler,      'model/scaler.pkl')
joblib.dump(le,          'model/gender_encoder.pkl')

with open('model/feature_names.json', 'w') as f:
    json.dump(feature_names, f)

metadata = {
    'best_model': 'Random Forest (Tuned)',
    'target_mapping': {'Liver Disease': 1, 'No Liver Disease': 0},
    'source_csv': csv_path
}
with open('model/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Saved files:')
print(' - model/liver_model.pkl')
print(' - model/scaler.pkl')
print(' - model/gender_encoder.pkl')
print(' - model/feature_names.json')
print(' - model/model_metadata.json')

---

## Project Summary

| Task | Status |
|---|---|
| Task 1 — Complete Data Analysis Report (EDA) | Done |
| Task 2 — 8 Classifiers with full evaluation | Done |
| Task 3 — Model design basis and analysis | Done |
| Model Comparison Report | Done |
| Challenges Report with techniques and reasons | Done |
| Best Model saved to disk | Done |

**Best Model:** Random Forest (Tuned) 
**Top Predictive Features:** Total Bilirubin, Direct Bilirubin, ALT, AST  
**Next Step:** Run `streamlit run app.py` to launch the prediction web app